# 06 Compare NER Backbones and Long-Report Methods

This notebook performs controlled NER experiments while keeping the report split and
random seed fixed. It supports both the original single-window truncation baseline and
overlapping sliding windows.

Recommended next run:

```text
PubMedBERT + 512-subword sliding windows + 128-subword overlap
```

Existing baseline files are not overwritten. Sliding-window outputs use the suffix
`_sliding_window`. Model selection uses validation F1, while the dissertation table
uses strict full-report entity metrics computed against the same gold reports.


In [ ]:
from __future__ import annotations

import inspect
import json
import math
import os
import sys
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from dotenv import load_dotenv
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from tqdm.auto import tqdm
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    set_seed,
)


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing pyproject.toml")


def format_duration(seconds: float) -> str:
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

RUN_NAME = os.getenv("RADGRAPH_XL_RUN_NAME", "full_2300")
RUN_ROOT = PROJECT_ROOT / "outputs" / RUN_NAME
INTERIM_DIR = RUN_ROOT / "interim"
RESULTS_DIR = RUN_ROOT / "results"
PREDICTIONS_DIR = RUN_ROOT / "predictions"
MODELS_DIR = RUN_ROOT / "models"
FIGURES_DIR = RUN_ROOT / "figures"

for path in [RESULTS_DIR, PREDICTIONS_DIR, MODELS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

NER_JSONL = INTERIM_DIR / "ner_dataset.jsonl"
LABEL_MAPS_JSON = INTERIM_DIR / "label_maps.json"
ENTITIES_CSV = INTERIM_DIR / "entities.csv"
assert NER_JSONL.exists(), "Run notebook 03 before notebook 06."
assert LABEL_MAPS_JSON.exists(), "Missing label_maps.json. Run notebook 03."
assert ENTITIES_CSV.exists(), "Missing entities.csv. Run notebook 03."

BACKBONES = {
    "bert_base_uncased": "bert-base-uncased",
    "pubmedbert": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
    "biobert": "dmis-lab/biobert-base-cased-v1.2",
    "bioclinicalbert": "emilyalsentzer/Bio_ClinicalBERT",
}

# Start with PubMedBERT because its existing truncated model is the stronger NER
# backbone. Add "bert_base_uncased" for a fully controlled sliding-window comparison.
EXPERIMENT_NAMES = ["pubmedbert"]

# "truncation" reproduces the old method. "sliding_window" covers the complete report.
LONG_REPORT_METHOD = "sliding_window"

# BERT-family encoders accept at most 512 subword tokens including special tokens.
MAX_LENGTH = 512

# Subword positions repeated between adjacent windows. The effective stride is
# MAX_LENGTH - WINDOW_OVERLAP, apart from special-token handling by the tokenizer.
WINDOW_OVERLAP = 128

# All training and model-comparison experiments keep the original fixed seed.
RANDOM_SEED = 42

LEARNING_RATE = 2e-5
NUM_EPOCHS = 6
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
WEIGHT_DECAY = 0.01
LR_SCHEDULER_TYPE = "linear"
WARMUP_RATIO = 0.0
GRADIENT_ACCUMULATION_STEPS = 1
MAX_GRAD_NORM = 1.0
LOGGING_STEPS = 10
EARLY_STOPPING_PATIENCE = 2
USE_CLASS_WEIGHTED_LOSS = False
CLASS_WEIGHT_CAP = 5.0
USE_FP16 = False

# Reuse a completed best_model when a later export or evaluation step failed.
# This avoids repeating transformer training after the model was already saved.
REUSE_SAVED_MODEL = True

RUN_SMOKE_TEST = False
MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = None
MAX_TEST_SAMPLES = None

if RUN_SMOKE_TEST:
    NUM_EPOCHS = 1
    MAX_TRAIN_SAMPLES = 16
    MAX_EVAL_SAMPLES = 8
    MAX_TEST_SAMPLES = 8

assert LONG_REPORT_METHOD in {"truncation", "sliding_window"}
assert 0 <= WINDOW_OVERLAP < MAX_LENGTH - 2
unknown_experiments = sorted(set(EXPERIMENT_NAMES) - set(BACKBONES))
assert not unknown_experiments, f"Unknown experiment names: {unknown_experiments}"

set_seed(RANDOM_SEED)

environment = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "python_executable": sys.executable,
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "experiments": EXPERIMENT_NAMES,
    "long_report_method": LONG_REPORT_METHOD,
    "max_length": MAX_LENGTH,
    "window_overlap": WINDOW_OVERLAP if LONG_REPORT_METHOD == "sliding_window" else 0,
    "random_seed": RANDOM_SEED,
    "reuse_saved_model": REUSE_SAVED_MODEL,
    "run_smoke_test": RUN_SMOKE_TEST,
}
print("Environment and experiment configuration")
for key, value in environment.items():
    print(f"  {key}: {value}")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select the project's .venv as the notebook kernel.")


In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as handle:
        total = sum(1 for line in handle if line.strip())
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in tqdm(handle, total=total, desc="Loading NER reports", unit="report"):
            if line.strip():
                rows.append(json.loads(line))
    return rows


rows = load_jsonl(NER_JSONL)
canonical_entities = pd.read_csv(ENTITIES_CSV)
canonical_test_entities = canonical_entities[
    canonical_entities["split"] == "test"
].copy()
label_maps = json.loads(LABEL_MAPS_JSON.read_text(encoding="utf-8"))
label_to_id = {label: int(idx) for label, idx in label_maps["ner_label_to_id"].items()}
id_to_label = {int(idx): label for idx, label in label_maps["ner_id_to_label"].items()}

splits = {
    split: [row for row in rows if row["split"] == split]
    for split in ["train", "validation", "test"]
}

if MAX_TRAIN_SAMPLES is not None:
    splits["train"] = splits["train"][:MAX_TRAIN_SAMPLES]
if MAX_EVAL_SAMPLES is not None:
    splits["validation"] = splits["validation"][:MAX_EVAL_SAMPLES]
if MAX_TEST_SAMPLES is not None:
    splits["test"] = splits["test"][:MAX_TEST_SAMPLES]

raw_dataset = DatasetDict({split: Dataset.from_list(items) for split, items in splits.items()})

label_counts = Counter(label for row in splits["train"] for label in row["bio_labels"])
raw_weights = np.array(
    [
        math.sqrt(sum(label_counts.values()) / max(1, len(label_counts) * label_counts[label]))
        for label in label_to_id
    ],
    dtype=np.float32,
)
raw_weights = raw_weights / raw_weights.mean()
raw_weights = np.minimum(raw_weights, CLASS_WEIGHT_CAP)
class_weights_tensor = torch.tensor(raw_weights, dtype=torch.float32)

print("Report split sizes:", {split: len(items) for split, items in splits.items()})
print("BIO labels:", len(label_to_id))
print("Class weighting enabled:", USE_CLASS_WEIGHTED_LOSS)
if USE_CLASS_WEIGHTED_LOSS:
    print({label: round(float(raw_weights[idx]), 3) for label, idx in label_to_id.items()})


In [ ]:
def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []
    for pred_row, label_row in zip(predictions, labels):
        pred_labels = []
        gold_labels = []
        for pred_id, label_id in zip(pred_row, label_row):
            if label_id == -100:
                continue
            pred_labels.append(id_to_label[int(pred_id)])
            gold_labels.append(id_to_label[int(label_id)])
        true_predictions.append(pred_labels)
        true_labels.append(gold_labels)

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }


def bio_to_spans(labels: list[str]) -> list[dict]:
    spans = []
    start = None
    current_label = None

    def close_span(end_index: int) -> None:
        if start is not None and current_label is not None:
            spans.append({"start": start, "end": end_index, "label": current_label})

    for index, label in enumerate(labels):
        if label == "O":
            if current_label is not None:
                close_span(index - 1)
                start = None
                current_label = None
            continue
        prefix, entity_label = label.split("-", 1)
        if prefix == "B" or current_label != entity_label:
            if current_label is not None:
                close_span(index - 1)
            start = index
            current_label = entity_label

    if current_label is not None:
        close_span(len(labels) - 1)
    return spans


def predict_word_labels(model, tokenizer, row: dict) -> list[str]:
    tokenizer_kwargs = {
        "is_split_into_words": True,
        "truncation": True,
        "max_length": MAX_LENGTH,
        # Overflow windows have different final lengths. Padding is required
        # before Hugging Face can stack them into one PyTorch tensor.
        "padding": True,
        "return_tensors": "pt",
    }
    if LONG_REPORT_METHOD == "sliding_window":
        tokenizer_kwargs.update(
            {
                "return_overflowing_tokens": True,
                "stride": WINDOW_OVERLAP,
            }
        )

    encoded = tokenizer(row["tokens"], **tokenizer_kwargs)
    word_ids_by_window = [
        encoded.word_ids(batch_index=window_index)
        for window_index in range(encoded["input_ids"].shape[0])
    ]
    encoded.pop("overflow_to_sample_mapping", None)
    model_inputs = {key: value.to(model.device) for key, value in encoded.items()}

    model.eval()
    with torch.no_grad():
        probabilities = torch.softmax(model(**model_inputs).logits, dim=-1).cpu().numpy()

    score_sums = np.zeros((len(row["tokens"]), len(id_to_label)), dtype=np.float64)
    score_counts = np.zeros(len(row["tokens"]), dtype=np.int32)

    for window_index, word_ids in enumerate(word_ids_by_window):
        previous_word_id = None
        for token_index, word_id in enumerate(word_ids):
            if word_id is None or word_id == previous_word_id:
                previous_word_id = word_id
                continue
            score_sums[word_id] += probabilities[window_index, token_index]
            score_counts[word_id] += 1
            previous_word_id = word_id

    predicted_labels = []
    for word_index in range(len(row["tokens"])):
        if score_counts[word_index] == 0:
            predicted_labels.append("O")
        else:
            mean_scores = score_sums[word_index] / score_counts[word_index]
            predicted_labels.append(id_to_label[int(mean_scores.argmax())])
    return predicted_labels


def strict_entity_metrics(prediction_rows: list[dict]) -> dict:
    prediction_doc_ids = {row["doc_id"] for row in prediction_rows}
    gold_frame = canonical_test_entities[
        canonical_test_entities["doc_id"].isin(prediction_doc_ids)
    ]
    gold = {
        (row.doc_id, int(row.start), int(row.end), row.safe_label)
        for row in gold_frame.itertuples(index=False)
    }
    predicted = {
        (row["doc_id"], int(entity["start"]), int(entity["end"]), entity["label"])
        for row in prediction_rows
        for entity in row["predicted_entities"]
    }

    def scores(gold_set: set[tuple], predicted_set: set[tuple]) -> tuple[float, float, float]:
        tp = len(gold_set & predicted_set)
        fp = len(predicted_set - gold_set)
        fn = len(gold_set - predicted_set)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        return precision, recall, f1

    precision, recall, micro_f1 = scores(gold, predicted)
    entity_labels = sorted({item[-1] for item in gold | predicted})
    per_label = {}
    for label in entity_labels:
        label_gold = {item for item in gold if item[-1] == label}
        label_predicted = {item for item in predicted if item[-1] == label}
        label_precision, label_recall, label_f1 = scores(label_gold, label_predicted)
        per_label[label] = {
            "precision": label_precision,
            "recall": label_recall,
            "f1": label_f1,
            "gold": len(label_gold),
            "predicted": len(label_predicted),
        }

    return {
        "precision": precision,
        "recall": recall,
        "micro_f1": micro_f1,
        "macro_f1": float(np.mean([row["f1"] for row in per_label.values()])),
        "gold": len(gold),
        "predicted": len(predicted),
        "true_positives": len(gold & predicted),
        "per_label": per_label,
    }


In [ ]:
class ProgressPrinterCallback(TrainerCallback):
    def __init__(self, experiment_name: str) -> None:
        self.experiment_name = experiment_name
        self.started_at = None
        self.last_logged_step = -1

    def on_train_begin(self, args, state, control, **kwargs):
        self.started_at = time.perf_counter()
        print(f"[{self.experiment_name}] training started: {state.max_steps} steps on {args.device}")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or state.global_step == self.last_logged_step:
            return
        self.last_logged_step = state.global_step
        elapsed = time.perf_counter() - self.started_at if self.started_at else 0
        percent = 100 * state.global_step / max(1, state.max_steps)
        values = []
        for key in ["loss", "eval_loss", "eval_f1", "learning_rate"]:
            if key in logs:
                values.append(f"{key}={logs[key]:.6g}")
        print(
            f"[{self.experiment_name}] step {state.global_step}/{state.max_steps} "
            f"({percent:.1f}%), epoch={state.epoch or 0:.2f}, "
            f"elapsed={format_duration(elapsed)}, " + ", ".join(values)
        )

    def on_train_end(self, args, state, control, **kwargs):
        elapsed = time.perf_counter() - self.started_at if self.started_at else 0
        print(f"[{self.experiment_name}] training finished in {format_duration(elapsed)}")


class WeightedTokenTrainer(Trainer):
    def __init__(self, *args, class_weights: torch.Tensor | None = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = torch.nn.functional.cross_entropy(
            logits.view(-1, model.config.num_labels),
            labels.view(-1),
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None,
            ignore_index=-100,
        )
        return (loss, outputs) if return_outputs else loss


def training_arguments(model_dir: Path) -> TrainingArguments:
    kwargs = {
        "output_dir": str(model_dir),
        "learning_rate": LEARNING_RATE,
        "per_device_train_batch_size": TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": EVAL_BATCH_SIZE,
        "num_train_epochs": NUM_EPOCHS,
        "weight_decay": WEIGHT_DECAY,
        "lr_scheduler_type": LR_SCHEDULER_TYPE,
        "warmup_ratio": WARMUP_RATIO,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "max_grad_norm": MAX_GRAD_NORM,
        "save_strategy": "epoch",
        "save_total_limit": 2,
        "logging_strategy": "steps",
        "logging_steps": LOGGING_STEPS,
        "logging_first_step": True,
        "disable_tqdm": False,
        "load_best_model_at_end": True,
        "metric_for_best_model": "f1",
        "greater_is_better": True,
        "report_to": [],
        "seed": RANDOM_SEED,
        "dataloader_num_workers": 0,
        "fp16": USE_FP16 and torch.cuda.is_available(),
    }
    signature = inspect.signature(TrainingArguments.__init__)
    if "eval_strategy" in signature.parameters:
        kwargs["eval_strategy"] = "epoch"
    else:
        kwargs["evaluation_strategy"] = "epoch"
    return TrainingArguments(**kwargs)

In [ ]:
experiment_rows = []

for experiment_name in EXPERIMENT_NAMES:
    checkpoint = BACKBONES[experiment_name]
    run_key = (
        experiment_name
        if LONG_REPORT_METHOD == "truncation"
        else f"{experiment_name}_sliding_window"
    )
    model_dir = MODELS_DIR / f"ner_{run_key}"
    best_model_dir = model_dir / "best_model"

    print("\n" + "=" * 80)
    print(f"Experiment: {run_key}")
    print(f"Checkpoint: {checkpoint}")

    reuse_saved_model = (
        REUSE_SAVED_MODEL
        and (best_model_dir / "config.json").exists()
        and (best_model_dir / "model.safetensors").exists()
    )
    model_source = best_model_dir if reuse_saved_model else checkpoint

    load_started = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(model_source, use_fast=True)
    print(f"Tokenizer loaded from {model_source} in "
          f"{format_duration(time.perf_counter() - load_started)}")

    def tokenize_and_align_labels(batch: dict) -> dict:
        tokenizer_kwargs = {
            "is_split_into_words": True,
            "truncation": True,
            "max_length": MAX_LENGTH,
        }
        if LONG_REPORT_METHOD == "sliding_window":
            tokenizer_kwargs.update(
                {
                    "return_overflowing_tokens": True,
                    "stride": WINDOW_OVERLAP,
                }
            )

        tokenized = tokenizer(batch["tokens"], **tokenizer_kwargs)
        if LONG_REPORT_METHOD == "sliding_window":
            sample_mapping = list(tokenized.pop("overflow_to_sample_mapping"))
        else:
            sample_mapping = list(range(len(batch["tokens"])))

        aligned_labels = []
        for encoded_index, sample_index in enumerate(sample_mapping):
            bio_labels = batch["bio_labels"][sample_index]
            word_ids = tokenized.word_ids(batch_index=encoded_index)
            previous_word_id = None
            label_ids = []
            for word_id in word_ids:
                if word_id is None:
                    label_ids.append(-100)
                elif word_id != previous_word_id:
                    label_ids.append(label_to_id[bio_labels[word_id]])
                else:
                    label_ids.append(-100)
                previous_word_id = word_id
            aligned_labels.append(label_ids)
        tokenized["labels"] = aligned_labels
        return tokenized

    tokenized_dataset = DatasetDict(
        {
            split_name: split_dataset.map(
                tokenize_and_align_labels,
                batched=True,
                remove_columns=split_dataset.column_names,
                desc=f"{run_key}: tokenising {split_name}",
            )
            for split_name, split_dataset in raw_dataset.items()
        }
    )

    if reuse_saved_model:
        model = AutoModelForTokenClassification.from_pretrained(
            best_model_dir,
        )
        print(f"Reusing saved model: {best_model_dir}")
    else:
        model = AutoModelForTokenClassification.from_pretrained(
            checkpoint,
            num_labels=len(label_to_id),
            id2label=id_to_label,
            label2id=label_to_id,
        )
    args = training_arguments(model_dir)
    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

    trainer_kwargs = {
        "model": model,
        "args": args,
        "train_dataset": tokenized_dataset["train"],
        "eval_dataset": tokenized_dataset["validation"],
        "data_collator": data_collator,
        "compute_metrics": compute_metrics,
        "callbacks": [
            ProgressPrinterCallback(run_key),
            EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
        ],
        "class_weights": class_weights_tensor if USE_CLASS_WEIGHTED_LOSS else None,
    }
    if "processing_class" in inspect.signature(Trainer.__init__).parameters:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = WeightedTokenTrainer(**trainer_kwargs)
    steps_per_epoch = math.ceil(len(tokenized_dataset["train"]) / TRAIN_BATCH_SIZE)
    print(
        {
            "train_reports": len(splits["train"]),
            "train_windows": len(tokenized_dataset["train"]),
            "validation_reports": len(splits["validation"]),
            "validation_windows": len(tokenized_dataset["validation"]),
            "maximum_epochs": NUM_EPOCHS,
            "estimated_maximum_steps": steps_per_epoch * NUM_EPOCHS,
            "device": str(args.device),
        }
    )

    if reuse_saved_model:
        train_metrics = {
            "training_skipped": True,
            "reused_model": str(best_model_dir),
        }
        print("Training skipped because best_model is already complete.")
    else:
        train_result = trainer.train()
        train_metrics = train_result.metrics
        trainer.save_model(str(best_model_dir))
        tokenizer.save_pretrained(str(best_model_dir))

    window_test_metrics = trainer.evaluate(
        tokenized_dataset["test"],
        metric_key_prefix="test_window",
    )

    prediction_rows = []
    prediction_path = PREDICTIONS_DIR / f"ner_{run_key}_test_predictions.jsonl"
    with prediction_path.open("w", encoding="utf-8") as handle:
        for row in tqdm(
            splits["test"],
            desc=f"{run_key}: exporting full-report predictions",
            unit="report",
        ):
            predicted_labels = predict_word_labels(model, tokenizer, row)
            output = {
                "doc_id": row["doc_id"],
                "split": row["split"],
                "predicted_entities": bio_to_spans(predicted_labels),
                "gold_entities": bio_to_spans(row["bio_labels"]),
            }
            prediction_rows.append(output)
            handle.write(json.dumps(output, ensure_ascii=False) + "\n")

    full_report_metrics = strict_entity_metrics(prediction_rows)
    full_report_metrics.update(
        {
            "experiment_name": run_key,
            "model_checkpoint": checkpoint,
            "long_report_method": LONG_REPORT_METHOD,
            "max_length": MAX_LENGTH,
            "window_overlap": WINDOW_OVERLAP if LONG_REPORT_METHOD == "sliding_window" else 0,
            "random_seed": RANDOM_SEED,
        }
    )

    window_metrics_path = RESULTS_DIR / f"ner_{run_key}_metrics.json"
    window_metrics_path.write_text(json.dumps(window_test_metrics, indent=2), encoding="utf-8")
    full_metrics_path = RESULTS_DIR / f"ner_{run_key}_full_report_metrics.json"
    full_metrics_path.write_text(json.dumps(full_report_metrics, indent=2), encoding="utf-8")

    run_config = {
        "experiment_name": run_key,
        "backbone_name": experiment_name,
        "model_checkpoint": checkpoint,
        "long_report_method": LONG_REPORT_METHOD,
        "max_length": MAX_LENGTH,
        "window_overlap": WINDOW_OVERLAP if LONG_REPORT_METHOD == "sliding_window" else 0,
        "effective_window_advance": (
            MAX_LENGTH - WINDOW_OVERLAP
            if LONG_REPORT_METHOD == "sliding_window"
            else None
        ),
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "weight_decay": WEIGHT_DECAY,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "use_class_weighted_loss": USE_CLASS_WEIGHTED_LOSS,
        "class_weight_cap": CLASS_WEIGHT_CAP if USE_CLASS_WEIGHTED_LOSS else None,
        "use_fp16": USE_FP16,
        "random_seed": RANDOM_SEED,
        "run_smoke_test": RUN_SMOKE_TEST,
        "training_reused_from_saved_model": reuse_saved_model,
        "report_counts": {split: len(items) for split, items in splits.items()},
        "window_counts": {
            split: len(tokenized_dataset[split])
            for split in ["train", "validation", "test"]
        },
        "train_metrics": train_metrics,
        "primary_evaluation": "strict full-report entity matching",
    }
    config_path = RESULTS_DIR / f"ner_{run_key}_run_config.json"
    config_path.write_text(json.dumps(run_config, indent=2), encoding="utf-8")

    report_predictions = trainer.predict(tokenized_dataset["test"])
    report_pred_ids = np.argmax(report_predictions.predictions, axis=-1)
    report_label_ids = report_predictions.label_ids
    all_predicted_labels = []
    all_gold_labels = []
    for pred_row, label_row in zip(report_pred_ids, report_label_ids):
        pred_labels = []
        gold_labels = []
        for pred_id, label_id in zip(pred_row, label_row):
            if label_id == -100:
                continue
            pred_labels.append(id_to_label[int(pred_id)])
            gold_labels.append(id_to_label[int(label_id)])
        all_predicted_labels.append(pred_labels)
        all_gold_labels.append(gold_labels)

    report = classification_report(all_gold_labels, all_predicted_labels, digits=4)
    report_path = RESULTS_DIR / f"ner_{run_key}_window_classification_report.txt"
    report_path.write_text(report, encoding="utf-8")

    experiment_rows.append(
        {
            "experiment": run_key,
            "checkpoint": checkpoint,
            "long_report_method": LONG_REPORT_METHOD,
            "precision": full_report_metrics["precision"],
            "recall": full_report_metrics["recall"],
            "micro_f1": full_report_metrics["micro_f1"],
            "macro_f1": full_report_metrics["macro_f1"],
            "gold_entities": full_report_metrics["gold"],
        }
    )
    print(json.dumps(full_report_metrics, indent=2))
    print(
        "Saved:",
        window_metrics_path,
        full_metrics_path,
        config_path,
        prediction_path,
        report_path,
        sep="\n  ",
    )

    del trainer, model, tokenizer, tokenized_dataset
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
comparison_rows = []

for metrics_path in sorted(RESULTS_DIR.glob("ner_*_full_report_metrics.json")):
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    experiment_name = metrics.get("experiment_name")
    if not experiment_name:
        experiment_name = (
            metrics_path.name.removeprefix("ner_").removesuffix("_full_report_metrics.json")
        )
    config_path = RESULTS_DIR / f"ner_{experiment_name}_run_config.json"
    config = (
        json.loads(config_path.read_text(encoding="utf-8"))
        if config_path.exists()
        else {}
    )
    comparison_rows.append(
        {
            "experiment": experiment_name,
            "checkpoint": metrics.get("model_checkpoint", config.get("model_checkpoint")),
            "long_report_method": metrics.get(
                "long_report_method",
                config.get("long_report_method", "truncation"),
            ),
            "precision": metrics.get("precision"),
            "recall": metrics.get("recall"),
            "micro_f1": metrics.get("micro_f1"),
            "macro_f1": metrics.get("macro_f1"),
            "gold_entities": metrics.get(
                "gold", metrics.get("gold_entities")
            ),
            "random_seed": metrics.get("random_seed", config.get("random_seed")),
        }
    )

# Existing prediction files may predate the full-report metric export. Recompute them
# against their own embedded complete gold entity lists so they remain comparable.
known_experiments = {row["experiment"] for row in comparison_rows}
for prediction_path in sorted(PREDICTIONS_DIR.glob("ner_*_test_predictions.jsonl")):
    experiment_name = prediction_path.name.removeprefix("ner_").removesuffix(
        "_test_predictions.jsonl"
    )
    if experiment_name in known_experiments:
        continue
    prediction_rows = []
    with prediction_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                prediction_rows.append(json.loads(line))
    metrics = strict_entity_metrics(prediction_rows)
    config_path = RESULTS_DIR / f"ner_{experiment_name}_run_config.json"
    config = (
        json.loads(config_path.read_text(encoding="utf-8"))
        if config_path.exists()
        else {}
    )
    metrics.update(
        {
            "experiment_name": experiment_name,
            "model_checkpoint": config.get("model_checkpoint"),
            "long_report_method": config.get("long_report_method", "truncation"),
            "random_seed": config.get("random_seed", RANDOM_SEED),
        }
    )
    full_metrics_path = RESULTS_DIR / f"ner_{experiment_name}_full_report_metrics.json"
    full_metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    comparison_rows.append(
        {
            "experiment": experiment_name,
            "checkpoint": metrics.get("model_checkpoint"),
            "long_report_method": metrics["long_report_method"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "micro_f1": metrics["micro_f1"],
            "macro_f1": metrics["macro_f1"],
            "gold_entities": metrics["gold"],
            "random_seed": metrics["random_seed"],
        }
    )

comparison = pd.DataFrame(comparison_rows).drop_duplicates("experiment", keep="last")
comparison_path = RESULTS_DIR / "ner_full_report_backbone_comparison.csv"
comparison.to_csv(comparison_path, index=False)
display(comparison.sort_values("micro_f1", ascending=False))

if not comparison.empty:
    plot_data = comparison.sort_values("micro_f1", ascending=True)
    fig, ax = plt.subplots(figsize=(9, max(3.5, 0.7 * len(plot_data))))
    ax.barh(plot_data["experiment"], plot_data["micro_f1"], color="#3A7D44")
    ax.set_xlim(0, 1)
    ax.set_xlabel("Strict full-report entity micro F1")
    ax.set_ylabel("")
    ax.set_title("NER Backbone and Long-Report Method Comparison")
    ax.grid(axis="x", alpha=0.2)
    fig.tight_layout()
    figure_path = FIGURES_DIR / "ner_full_report_comparison_f1.png"
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved figure:", figure_path)

print("Saved comparison:", comparison_path)
print("All rows use complete report-level gold entity sets. Random seed remains 42.")
